In [ ]:
import pandas as pd

In [ ]:
PATH = "../data/deep learning project data/"
INVS = "invoices.csv"
ORDS = "orders.csv"
EMFS = "EMFs.csv"
OUT  = "documents.csv"

VAL    = "volume"
FYEAR  = "fund year"
YEAR   = "year"
MONTH  = "month"
GROUP  = ["doc", "tressure group"]

In [ ]:
invoices = pd.read_csv(PATH + INVS,  usecols = [VAL, YEAR, MONTH,  * GROUP])
orders   = pd.read_csv(PATH + ORDS,  usecols = [VAL, YEAR, MONTH,  * GROUP, FYEAR])
emfs     = pd.read_csv(PATH + EMFS,  usecols = [VAL, YEAR, MONTH,  * GROUP, FYEAR])

In [ ]:
def update_dates(table: pd.DataFrame) -> pd.DataFrame:

    mask = table[YEAR] < table[FYEAR]
    table.loc[mask, YEAR] = table.loc[mask, FYEAR]
    table.loc[mask, MONTH] = 1
    table = table.drop(columns = [FYEAR])
    
    return table


def unique_key(table: pd.DataFrame) -> pd.DataFrame:

    return table.groupby(table.columns.difference([VAL]).tolist(), as_index = False)[VAL].sum()


def accumulate(table: pd.DataFrame) -> pd.DataFrame:

    table = table.sort_values(by = [* GROUP, YEAR, MONTH])
    table[VAL] = table.groupby([ * GROUP, YEAR])[VAL].cumsum()

    return table


def clip(table: pd.DataFrame) -> pd.DataFrame:

    table = table[table[YEAR].between(invoices[YEAR].min(), invoices[YEAR].max())]
    table = table[(table[YEAR] < invoices[YEAR].max()) | (table[MONTH] <= invoices[MONTH].max())]

    return table


def pipeline(table: pd.DataFrame) -> pd.DataFrame:

    table = update_dates(table)
    table = unique_key(table)
    table = accumulate(table)
    table = clip(table)

    return table



emfs = pipeline(emfs)
orders = pipeline(orders)

In [ ]:
orders.groupby([YEAR, MONTH])[VAL].sum().plot(title = "Orders")
invoices.groupby([YEAR, MONTH])[VAL].sum().plot(title = "Invoices")

In [ ]:
combined = pd.concat([invoices, orders, emfs], ignore_index = True)

In [ ]:
combined.head()

In [ ]:
combined.tail()

In [ ]:
combined.to_csv(PATH + OUT, index = False)